# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="NutritionAssistant",
    instructions="""AgentYou are a helpful assistant giving out nutrition advice."
    You give concise answers."""
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="NutritionAssistant", ...)
- Final output (str):
    Overall, bananas are a healthy, convenient fruit.
    
    Key points:
    - Nutrients: potassium, vitamin C, vitamin B6, fiber, and resistant starch.
    - Benefits: supports heart health, digestion, and energy; helps with blood pressure regulation.
    - Sugar and calories: moderate; a medium banana has about 100 calories and natural sugars.
    - Glycemic index: moderate; ripe bananas are slightly higher in sugars.
    - Considerations: great for most people; those with kidney disease or potassium-sensitive conditions should watch intake.
    
    If you want, I can tailor advice to your goals (weight, diabetes, CKD, etc.).
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, nutrient-dense fruit.

Key nutrients per medium banana:
- Potassium
- Vitamin C
- Vitamin B6
- Dietary fiber
- Quick-source of energy (natural sugars)

Health benefits:
- Supports heart health and digestion
- Helps with satiety and energy

Notes:
- Moderate portion: about 105 calories; sugar content varies with ripeness (ripe = a bit sweeter).
- Diabetics or kidney issues: watch portions due to sugars or potassium; consult a clinician if needed.
- Green bananas have resistant starch; ripe bananas have more sugar.

Want a quick serving suggestion or portion guide?

_Good Job!_